In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

# 1. 가상 데이터셋 생성 (3개 군집, 2차원 공간)
X, y_true = make_blobs(
    n_samples=300, centers=3, cluster_std=0.60, random_state=42
)

# 2. K-Means 모델 정의 및 학습 (K=3, K-Means++ 초기화)
kmeans = KMeans(n_clusters=3, init="k-means++", n_init=10, random_state=42)
labels = kmeans.fit_predict(X)
centroids = kmeans.cluster_centers_

# ==================================================
# 3. 단일 K(K=3)에 대한 평가 지표 계산
# ==================================================
inertia = kmeans.inertia_  # 군집 내 오차제곱합 (SSE)
sil_score = silhouette_score(X, labels)
ch_score = calinski_harabasz_score(X, labels)
db_score = davies_bouldin_score(X, labels)

print("=== K-Means 군집화 평가 결과 (K=3) ===")
print(f"1. Inertia (군집 내 SSE)         : {inertia:.4f}")
print(f"2. 실루엣 계수 (Silhouette Score) : {sil_score:.4f} (1에 가까울수록 양호)")
print(f"3. Calinski-Harabasz Index     : {ch_score:.4f} (높을수록 양호)")
print(f"4. Davies-Bouldin Index        : {db_score:.4f} (0에 가까울수록 양호)")
print("-" * 45)

# ==================================================
# 4. 최적 K 탐색 (Elbow Method & Silhouette Analysis)
# ==================================================
inertias = []
sil_scores = []
K_range = range(2, 7)

for k in K_range:
  km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
  cluster_labels = km.fit_predict(X)

  inertias.append(km.inertia_)
  sil_scores.append(silhouette_score(X, cluster_labels))

# 5. 시각화 (군집 결과 + 엘보우 그래프 + 실루엣 변화)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) K=3 군집화 산점도
axes[0].scatter(
    X[:, 0],
    X[:, 1],
    c=labels,
    s=40,
    cmap="viridis",
    alpha=0.7,
    label="Data Points",
)
axes[0].scatter(
    centroids[:, 0],
    centroids[:, 1],
    c="red",
    s=200,
    marker="X",
    label="Centroids",
)
axes[0].set_title("K-Means Clustering Result (K=3)", fontsize=12)
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# (2) Elbow Method (Inertia vs K)
axes[1].plot(K_range, inertias, "bo-", linewidth=2, markersize=8)
axes[1].set_title("Elbow Method (Inertia vs K)", fontsize=12)
axes[1].set_xlabel("Number of Clusters (K)")
axes[1].set_ylabel("Inertia (SSE)")
axes[1].grid(True, linestyle="--", alpha=0.5)

# (3) Silhouette Score vs K
axes[2].plot(K_range, sil_scores, "ro-", linewidth=2, markersize=8)
axes[2].set_title("Silhouette Score vs K", fontsize=12)
axes[2].set_xlabel("Number of Clusters (K)")
axes[2].set_ylabel("Silhouette Score")
axes[2].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()